<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](mlcourse.ai) – دورة مفتوحة للتعلم الآلي 
### <center> المؤلف: إيليا لارشينكو، ODS Slack ilya_l
    
## <center> مشروع تحليل البيانات الفردية



## 1. وصف البيانات



__سوف أقوم بتحليل بيانات الإسكان في كاليفورنيا (1990). يمكن تنزيله من Kaggle [https://www.kaggle.com/harrywang/housing]__



سوف نتنبأ بمتوسط سعر الأسرة في الكتلة. 
للبدء، تحتاج إلى تنزيل ملف Housing.csv.zip. دعونا تحميل البيانات وننظر إليها.


In [ ]:
import pandas as pd
import numpy as np
import os
%matplotlib inline

import warnings                                  # `do not disturbe` mode
warnings.filterwarnings('ignore')

In [ ]:
# change this if needed
PATH_TO_DATA = 'data'

In [ ]:
full_df = pd.read_csv(os.path.join(PATH_TO_DATA, 'housing.csv.zip'), compression ='zip')
print(full_df.shape)
full_df.head()


تتكون البيانات من 20640 صفًا و10 ميزات:



1. خط الطول: قياس مدى بعد المنزل غربًا؛ القيمة الأعلى تقع في أقصى الغرب
2. خط العرض: قياس مدى بعد المنزل شمالًا؛ القيمة الأعلى تقع في أقصى الشمال
3. متوسط عمر السكن: متوسط عمر المنزل داخل المبنى؛ الرقم الأقل هو مبنى أحدث
4. إجمالي الغرف: إجمالي عدد الغرف داخل المبنى
5. إجمالي غرف النوم: إجمالي عدد غرف النوم داخل المبنى
6. السكان: إجمالي عدد الأشخاص المقيمين داخل المبنى
7. الأسر: إجمالي عدد الأسر، مجموعة من الأشخاص المقيمين داخل وحدة منزلية للكتلة
8. الدخل المتوسط: متوسط الدخل للأسر المعيشية التي تقع ضمن مجموعة من المنازل (يقاس بعشرات الآلاف من الدولارات الأمريكية)
9. قيمة المنزل المتوسطة: قيمة المنزل المتوسطة للأسر المعيشية داخل المبنى (مقاسة بالدولار الأمريكي)
10. القرب من المحيط: موقع المنزل بالقرب من المحيط / البحر



*قيمة_المنزل_الوسيط* هي الميزة المستهدفة لدينا، وسوف نستخدم ميزات أخرى للتنبؤ بها.
وتتمثل المهمة في التنبؤ بتكلفة المنازل على وجه الخصوص (الوسيط) بناءً على معلومات موقع الكتل والبيانات الاجتماعية والديموغرافية الأساسية



لنقسم مجموعة البيانات إلى تدريب (75%) واختبار (25%).


In [ ]:
%%time
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(full_df,shuffle = True, test_size = 0.25, random_state=17)
train_df=train_df.copy()
test_df=test_df.copy()
print(train_df.shape)
print(test_df.shape)

سنقوم بإجراء جميع التحليلات الإضافية مع مجموعة الاختبار. ولكن سيتم إنشاء الميزات ومعالجتها في نفس الوقت على كلا المجموعتين.



## 2-3. تحليل البيانات الأولية / تحليل البيانات المرئية الأولية


In [ ]:
train_df.describe()

In [ ]:
train_df.info()


يمكننا أن نرى أن معظم الأعمدة لا تحتوي على قيم نان (باستثناء Total_bedrooms)، ومعظم الميزات لها تنسيق عائم، وميزة واحدة فقط قاطعة - Ocean_proximity.


In [ ]:
train_df[pd.isnull(train_df).any(axis=1)].head(10)


لا توجد أسباب واضحة لأن تكون بعض غرف النوم الإجمالية NaN. يبلغ عدد NaNs حوالي 1٪ من إجمالي مجموعة البيانات. ربما يمكننا فقط إسقاط هذه الصفوف أو ملؤها بقيم متوسطة/متوسطة، ولكن دعونا ننتظر بعض الوقت، ونتعامل مع الفراغات بعد تحليل البيانات الأولية بطريقة أكثر ذكاءً.



لنقم بإنشاء قائمة بأسماء الميزات الرقمية (ستكون مفيدة لاحقًا).


In [ ]:
numerical_features=list(train_df.columns)
numerical_features.remove('ocean_proximity')
numerical_features.remove('median_house_value')
print(numerical_features)


دعونا نلقي نظرة على توزيع الميزة المستهدفة


In [ ]:
train_df['median_house_value'].hist()


يمكننا أن نرى بصريًا أن التوزيع منحرف وغير طبيعي. ويبدو أيضًا أن القيم قد تم قصها في مكان ما بالقرب من 500000. يمكننا التحقق من ذلك رقميًا.


In [ ]:
max_target=train_df['median_house_value'].max()
print("The largest median value:",max_target)
print("The # of values, equal to the largest:", sum(train_df['median_house_value']==max_target))
print("The % of values, equal to the largest:", sum(train_df['median_house_value']==max_target)/train_df.shape[0])


ما يقرب من 5% من جميع القيم = بالضبط 500001. وهذا يثبت نظرية القطع لدينا. دعونا نتحقق من قص القيم الصغيرة:


In [ ]:
min_target=train_df['median_house_value'].min()
print("The smallest median value:",min_target)
print("The # of values, equal to the smallest:", sum(train_df['median_house_value']==min_target))
print("The % of values, equal to the smallest:", sum(train_df['median_house_value']==min_target)/train_df.shape[0])


هذه المرة تبدو أفضل بكثير، قيمة مصطنعة قليلاً 14999 - شائعة بالنسبة للأسعار. وهناك 4 فقط من هذه القيم. لذلك ربما لم يتم قص القيم الصغيرة.



دعونا نجري بعض الاختبارات الطبيعية:


In [ ]:
from statsmodels.graphics.gofplots import qqplot
from matplotlib import pyplot

qqplot(train_df['median_house_value'], line='s')
pyplot.show()

In [ ]:
from scipy.stats import normaltest

stat, p = normaltest(train_df['median_house_value'])
print('Statistics=%.3f, p=%.3f' % (stat, p))

alpha = 0.05
if p < alpha:  # null hypothesis: x comes from a normal distribution
    print("The null hypothesis can be rejected")
else:
    print("The null hypothesis cannot be rejected")


يُظهر اختبار QQ-plot واختبار D'Agostino وPearson الطبيعي أن التوزيع بعيد عن الطبيعي. يمكننا محاولة استخدام السجل (1+n) لجعله أكثر طبيعية:


In [ ]:
target_log=np.log1p(train_df['median_house_value'])
qqplot(target_log, line='s')
pyplot.show()

In [ ]:
stat, p = normaltest(target_log)
print('Statistics=%.3f, p=%.3f' % (stat, p))

alpha = 0.05
if p < alpha:  # null hypothesis: x comes from a normal distribution
    print("The null hypothesis can be rejected")
else:
    print("The null hypothesis cannot be rejected")

يبدو هذا الرسم البياني أفضل بكثير، حيث يتم قطع الأجزاء غير العادية الوحيدة بأسعار مرتفعة وأسعار منخفضة جدًا. لسوء الحظ، لا يمكننا إعادة بناء البيانات المقطوعة وإحصائيًا، لا يزال التوزيع غير طبيعي - القيمة p = 0، يمكن رفض الفرضية الصفرية للتوزيع الطبيعي.
على أي حال، يمكن أن يكون التنبؤ بـ target_log بدلاً من target خيارًا جيدًا بالنسبة لنا، ولكن لا يزال يتعين علينا التحقق منه أثناء مرحلة التحقق من صحة النموذج.


In [ ]:
train_df['median_house_value_log']=np.log1p(train_df['median_house_value'])
test_df['median_house_value_log']=np.log1p(test_df['median_house_value'])


الآن دعونا نحلل الميزات العددية. أولا وقبل كل شيء نحن بحاجة إلى إلقاء نظرة على توزيعاتهم.


In [ ]:
train_df[numerical_features].hist(bins=50, figsize=(10, 10))


بعض الميزات منحرفة بشكل كبير، ويجب أن تكون "خدعة السجل" الخاصة بنا مفيدة


In [ ]:
skewed_features=['households','median_income','population', 'total_bedrooms', 'total_rooms']
log_numerical_features=[]
for f in skewed_features:
    train_df[f + '_log']=np.log1p(train_df[f])
    test_df[f + '_log']=np.log1p(test_df[f])
    log_numerical_features.append(f + '_log')

In [ ]:
train_df[log_numerical_features].hist(bins=50, figsize=(10, 10))


تبدو ميزاتنا الجديدة أفضل بكثير (خلال مرحلة النمذجة يمكننا استخدام الميزات الأصلية أو الجديدة أو كليهما)



يبدو Housing_median_age مقطوعًا أيضًا. دعونا ننظر إلى أعلى قيمة لها على وجه التحديد.


In [ ]:
max_house_age=train_df['housing_median_age'].max()
print("The largest value:",max_house_age)
print("The # of values, equal to the largest:", sum(train_df['housing_median_age']==max_house_age))
print("The % of values, equal to the largest:", sum(train_df['housing_median_age']==max_house_age)/train_df.shape[0])


من المحتمل جدًا أن تكون البيانات قد تم قصها (هناك أيضًا احتمال ضئيل أنه في عام 1938 كان هناك مشروع إعادة إعمار كبير في كاليفورنيا ولكن يبدو أقل احتمالًا). لا يمكننا إعادة إنشاء القيم الأصلية، ولكن قد يكون من المفيد إنشاء قيمة ثنائية جديدة تشير إلى اقتصاص عمر المنزل.


In [ ]:
train_df['age_clipped']=train_df['housing_median_age']==max_house_age
test_df['age_clipped']=test_df['housing_median_age']==max_house_age


الآن سنقوم بتحليل الارتباط بين الميزات والمتغير المستهدف


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

corr_y = pd.DataFrame(train_df).corr()
plt.rcParams['figure.figsize'] = (20, 16)  # Размер картинок
sns.heatmap(corr_y, 
            xticklabels=corr_y.columns.values,
            yticklabels=corr_y.columns.values, annot=True)

يمكننا أن نرى بعض الأنماط (ربما واضحة) هنا:
    - ترتبط قيم المنازل بشكل كبير بمتوسط الدخل
    - عدد الأسر لا يرتبط بنسبة 100% بعدد السكان، يمكننا محاولة إضافة متوسط حجم الأسرة كميزة
    - يجب تحليل خط الطول وخط العرض بشكل منفصل (مجرد الارتباط مع المتغير المستهدف ليس مفيدًا جدًا)
    - هناك مجموعة من السمات المترابطة بشكل كبير: عدد الغرف، غرف النوم، عدد السكان، والأسر. قد يكون من المفيد تقليل أبعاد هذه المجموعة الفرعية، خاصة إذا استخدمنا النماذج الخطية
    - Total_bedrooms هي واحدة من هذه الميزات المترابطة للغاية، وهذا يعني أنه يمكننا ملء قيم NaN بدقة عالية باستخدام أبسط الانحدار الخطي



دعونا نحاول ملء NaNs بانحدار خطي بسيط:


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

lin = LinearRegression()

# we will train our model based on all numerical non-target features with not NaN total_bedrooms
appropriate_columns = train_df.drop(['median_house_value','median_house_value_log',
                            'ocean_proximity', 'total_bedrooms_log'],axis=1)
train_data=appropriate_columns[~pd.isnull(train_df).any(axis=1)]

# model will be validated on 25% of train dataset 
# theoretically we can use even our test_df dataset (as we don't use target) for this task, but we will not
temp_train, temp_valid = train_test_split(train_data,shuffle = True, test_size = 0.25, random_state=17)

lin.fit(temp_train.drop(['total_bedrooms'],axis=1), temp_train['total_bedrooms'])
np.sqrt(mean_squared_error(lin.predict(temp_valid.drop(['total_bedrooms'],axis=1)),
                           temp_valid['total_bedrooms']))


RMSE في مجموعة التحقق من الصحة هو 64.5. دعونا نقارن ذلك بأفضل تنبؤ ثابت - ماذا لو ملأنا NaNs بالقيمة المتوسطة:


In [ ]:
np.sqrt(mean_squared_error(np.ones(len(temp_valid['total_bedrooms']))*temp_train['total_bedrooms'].mean(),
                           temp_valid['total_bedrooms']))


من الواضح أن نهج الانحدار الخطي لدينا أفضل بكثير. دعونا ندرب نموذجنا على مجموعة بيانات القطار بأكملها ونطبقها على الصفوف التي تحتوي على فراغات. لكن مبدئيًا سوف "نتذكر" الصفوف التي تحتوي على NaNs، نظرًا لوجود احتمال أن تحتوي على معلومات مفيدة.


In [ ]:
lin.fit(train_data.drop(['total_bedrooms'],axis=1), train_data['total_bedrooms'])

train_df['total_bedrooms_is_nan']=pd.isnull(train_df).any(axis=1).astype(int)
test_df['total_bedrooms_is_nan']=pd.isnull(test_df).any(axis=1).astype(int)

train_df['total_bedrooms'].loc[pd.isnull(train_df).any(axis=1)]=\
lin.predict(train_df.drop(['median_house_value','median_house_value_log','total_bedrooms','total_bedrooms_log',
               'ocean_proximity','total_bedrooms_is_nan'],axis=1)[pd.isnull(train_df).any(axis=1)])

test_df['total_bedrooms'].loc[pd.isnull(test_df).any(axis=1)]=\
lin.predict(test_df.drop(['median_house_value','median_house_value_log','total_bedrooms','total_bedrooms_log',
               'ocean_proximity','total_bedrooms_is_nan'],axis=1)[pd.isnull(test_df).any(axis=1)])

#linear regression can lead to negative predictions, let's change it
test_df['total_bedrooms']=test_df['total_bedrooms'].apply(lambda x: max(x,0))
train_df['total_bedrooms']=train_df['total_bedrooms'].apply(lambda x: max(x,0))


فلنقم بتحديث "total_bedrooms_log" والتحقق من عدم وجود NaNs متبقية


In [ ]:
train_df['total_bedrooms_log']=np.log1p(train_df['total_bedrooms'])
test_df['total_bedrooms_log']=np.log1p(test_df['total_bedrooms'])

In [ ]:
print(train_df.info())
print(test_df.info())


بعد ملء الفراغات دعونا نلقي نظرة فاحصة على التبعيات بين بعض الميزات الرقمية


In [ ]:
sns.set()
sns.pairplot(train_df[log_numerical_features+['median_house_value_log']])


يبدو أنه لا توجد رؤى جديدة حول الميزات الرقمية (فقط تأكيد للميزات القديمة).
دعونا نحاول أن نفعل الشيء نفسه ولكن بالنسبة للمجموعة الفرعية المحلية (جغرافيًا) من بياناتنا.


In [ ]:
sns.set()
local_coord=[-122, 41] # the point near which we want to look at our variables
euc_dist_th = 2 # distance treshhold

euclid_distance=train_df[['latitude','longitude']].apply(lambda x:
                                                         np.sqrt((x['longitude']-local_coord[0])**2+
                                                                 (x['latitude']-local_coord[1])**2), axis=1)

# indicate wethere the point is within treshhold or not
indicator=pd.Series(euclid_distance<=euc_dist_th, name='indicator')

print("Data points within treshhold:", sum(indicator))

# a small map to visualize th eregion for analysis
sns.lmplot('longitude', 'latitude', data=pd.concat([train_df,indicator], axis=1), hue='indicator', markers ='.', fit_reg=False, height=5)

# pairplot
sns.pairplot(train_df[log_numerical_features+['median_house_value_log']][indicator])

يمكننا أن نرى أنه في أي منطقة محلية (يمكنك اللعب مع local_coord وeuc_dist_th) أصبحت الاعتمادات الخطية بين المتغيرات أقوى، وخاصة سجل الدخل المتوسط ​​/ سجل قيمة المنزل المتوسط. لذا فإن الإحداثيات عامل مهم جدًا لمهمتنا (سنقوم بتحليلها لاحقًا) 
الآن دعنا ننتقل إلى الميزة الفئوية "ocean_proximity". ليس من الواضح بنسبة 100% ما الذي تعنيه هذه القيم. لذلك دعونا أولاً نرسم الخريطة.


In [ ]:
sns.lmplot('longitude', 'latitude', data=train_df,markers ='.', hue='ocean_proximity', fit_reg=False, height=5)
plt.show()


الآن نحن نفهم بشكل أفضل معنى الفئات المختلفة. دعونا ننظر إلى البيانات.


In [ ]:
value_count=train_df['ocean_proximity'].value_counts()
value_count

In [ ]:
plt.figure(figsize=(12,5))


sns.barplot(value_count.index, value_count.values)
plt.title('Ocean Proximity')
plt.ylabel('Number of Occurrences')
plt.xlabel('Ocean Proximity')

plt.figure(figsize=(12,5))
plt.title('House Value depending on Ocean Proximity')
sns.boxplot(x="ocean_proximity", y="median_house_value_log", data=train_df)


يمكننا أن نرى أن أسعار المنازل الداخلية أقل بكثير. التوزيع في بلدان أخرى يختلف ولكن ليس كثيرا. لا يوجد اتجاه واضح في أسعار المنازل/القرب، لذلك لن نحاول ابتكار نهج تشفير معقد. دعونا نفعل فقط OHE لهذه الميزة.


In [ ]:
ocean_proximity_dummies = pd.get_dummies(pd.concat([train_df['ocean_proximity'],test_df['ocean_proximity']]),
                                         drop_first=True)

In [ ]:
dummies_names=list(ocean_proximity_dummies.columns)

In [ ]:
train_df=pd.concat([train_df,ocean_proximity_dummies[:train_df.shape[0]]], axis=1 )
test_df=pd.concat([test_df,ocean_proximity_dummies[train_df.shape[0]:]], axis=1 )

train_df=train_df.drop(['ocean_proximity'], axis=1)
test_df=test_df.drop(['ocean_proximity'], axis=1)

In [ ]:
train_df.head()


وأخيرا سوف نستكشف ميزات الإحداثيات.


In [ ]:
train_df[['longitude','latitude']].describe()


لنرسم قيم المنزل (الهدف) على الخريطة:


In [ ]:
from matplotlib.colors import LinearSegmentedColormap

plt.figure(figsize=(10,10))

cmap = LinearSegmentedColormap.from_list(name='name', colors=['green','yellow','red'])

f, ax = plt.subplots()
points = ax.scatter(train_df['longitude'], train_df['latitude'], c=train_df['median_house_value_log'],
                    s=10, cmap=cmap)
f.colorbar(points)


يبدو أن متوسط قيمة أقرب المنازل جغرافيًا يمكن أن يكون ميزة جيدة جدًا.
يمكننا أن نرى أيضًا أن أغلى المنازل تقع بالقرب من سان فرانسيسكو (37.7749 درجة شمالًا، 122.4194 درجة غربًا) ولوس أنجلوس (34.0522 درجة شمالًا، 118.2437 درجة). وبناءً على ذلك يمكننا استخدام المسافة إلى هذه المدن كميزات إضافية.
ونرى أيضًا أن أغلى المنازل تقع تقريبًا على الخط المستقيم، وتصبح أرخص عندما ننتقل إلى الشمال الشرقي. وهذا يعني أن الجمع الخطي للإحداثيات نفسها يمكن أن يكون ميزة مفيدة أيضًا.


In [ ]:
sf_coord=[-122.4194, 37.7749]
la_coord=[-118.2437, 34.0522]

train_df['distance_to_SF']=np.sqrt((train_df['longitude']-sf_coord[0])**2+(train_df['latitude']-sf_coord[1])**2)
test_df['distance_to_SF']=np.sqrt((test_df['longitude']-sf_coord[0])**2+(test_df['latitude']-sf_coord[1])**2)

train_df['distance_to_LA']=np.sqrt((train_df['longitude']-la_coord[0])**2+(train_df['latitude']-la_coord[1])**2)
test_df['distance_to_LA']=np.sqrt((test_df['longitude']-la_coord[0])**2+(test_df['latitude']-la_coord[1])**2)


## 4. الرؤى والتبعيات الموجودة


دعونا نلخص بسرعة ما وجدناه مفيدًا حتى الآن:
- لقد قمنا بتحليل الميزات ووجدنا بعض ~lognorm موزعة فيما بينها. لقد أنشأنا ميزات السجل المقابلة
- لقد قمنا بتحليل توزيع الميزة المستهدفة، وخلصنا إلى أنه قد يكون من المفيد التنبؤ بسجلها (سيتم التحقق منه)
- لقد تعاملنا مع البيانات المقطوعة والمفقودة
- لقد أنشأنا ميزات تتوافق مع المسافات الإقليدية البسيطة إلى لوس أنجلوس وسان فرانسيسكو
- لقد وجدنا أيضًا العديد من المتغيرات المترابطة للغاية وربما سنعمل معها لاحقًا
- لقد قمنا بالفعل بإنشاء العديد من المتغيرات الجديدة وسنقوم بإنشاء المزيد منها لاحقًا بعد مرحلة النمذجة الأولية
تم بالفعل تقديم كل التوضيحات حول هذه الخطوات أعلاه.



## 5. اختيار المقاييس



هذه هي مشكلة الانحدار. سيكون مقياسنا المستهدف هو RMSE - وهو أحد مقاييس الانحدار الأكثر شيوعًا، وله نفس وحدة قياس القيمة المستهدفة وبالتالي يسهل شرحه للآخرين. 
\بداية{محاذاة}
RMSE = \sqrt{\frac{1}{n}\Sigma_{i=1}^{n}{\Big(\frac{d_i -f_i}{\sigma_i}\Big)^2}}
\النهاية{محاذاة}
وبقدر ما يوجد اعتماد رتيب بين RMSE وMSE، يمكننا تحسين MSE في نموذجنا وحساب RMSE فقط في النهاية. من السهل تحسين MSE، فهي وظيفة خسارة افتراضية لمعظم نماذج الانحدار.
العيب الرئيسي في MSE وRMSE - العقوبة العالية للأخطاء الكبيرة في التنبؤات - يمكن أن يفرط في القيم المتطرفة، ولكن في حالتنا، تم بالفعل قص القيم المستهدفة المنفقة، لذا فهي ليست مشكلة كبيرة.



## 6. اختيار النموذج



سنحاول حل مشكلتنا باستخدام ثلاثة نماذج انحدار مختلفة:
- الانحدار الخطي
- غابة عشوائية
- تعزيز التدرجالانحدار الخطي سريع وبسيط ويمكن أن يوفر نتيجة أساسية جيدة لمهمتنا.
يمكن للنماذج المبنية على الشجرة أن توفر نتائج أفضل في حالة التبعيات المعقدة غير الخطية للمتغيرات، وفي حالة وجود عدد صغير من المتغيرات، فهي أيضًا أكثر استقرارًا بالنسبة للعلاقات الخطية المتعددة (ولدينا متغيرات مرتبطة بشكل كبير). علاوة على ذلك، في مشكلتنا، يتم قص القيم المستهدفة ولا يمكن أن تكون الأهداف خارج الفاصل الزمني للقص، وهذا أمر جيد بالنسبة للنماذج المستندة إلى الشجرة.
سيتم مقارنة نتائج استخدام هذه النماذج في الأجزاء 11-12 من المشروع. من المتوقع أن تعمل النماذج المبنية على الأشجار بشكل أفضل في هذه المشكلة تحديدًا، لكننا سنبدأ بنموذج أكثر بساطة.
سنبدأ بالانحدار الخطي القياسي، ونستعرض جميع خطوات النمذجة، ثم نقوم ببعض الحسابات المبسطة لنموذجين آخرين (بدون شرح متعمق لكل خطوة).
سيتم اختيار النموذج النهائي بناءً على النتائج.



## 7. المعالجة المسبقة للبيانات



لقد قمنا بالفعل بمعظم خطوات المعالجة المسبقة:
    - OHE للميزات الفئوية
    - نان مملوءة
    - السجلات المحسوبة للبيانات المنحرفة
    - تقسيم البيانات إلى مجموعات القطارات والإيقاف
الآن دعونا نقيس جميع الميزات العددية (وهو مفيد للنماذج الخطية)، ونقوم بإعداد تقسيمات التحقق المتقاطع ونحن على استعداد للشروع في النمذجة


In [ ]:
from sklearn.preprocessing import StandardScaler

features_to_scale=numerical_features+log_numerical_features+['distance_to_SF','distance_to_LA']

scaler = StandardScaler()

X_train_scaled=pd.DataFrame(scaler.fit_transform(train_df[features_to_scale]),
                            columns=features_to_scale, index=train_df.index)
X_test_scaled=pd.DataFrame(scaler.transform(test_df[features_to_scale]),
                           columns=features_to_scale, index=test_df.index)


## 8 التحقق من صحة وتعديل المعلمات الفائقة للنموذج 



دعونا نجهز عينات التحقق من الصحة.
وبقدر ما لا يوجد الكثير من البيانات، يمكننا تقسيمها بسهولة على 10 طيات، مأخوذة من بيانات القطار المختلطة.
وفي كل قسم، سنقوم بتدريب نموذجنا على 90% من بيانات القطار وحساب مقياس السيرة الذاتية على الـ 10% الأخرى.
نقوم بإصلاح الحالة العشوائية لإمكانية تكرار نتائج.


In [ ]:
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=10, random_state=17, shuffle=True)


### الانحدار الخطي


بالنسبة لخط الأساس الأولي الأول، سنأخذ نموذج Rigge مع الميزات العددية الأولية وخصائص OHE فقط


In [ ]:
from sklearn.linear_model import Ridge

model=Ridge(alpha=1)
X=train_df[numerical_features+dummies_names]
y=train_df['median_house_value']
cv_scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1)
print(np.sqrt(-cv_scores.mean()))


نحن نقوم بالتحقق المتقاطع بـ 10 طيات، ونحسب "neg_mean_squared_error" (neg - لأن sklearn يحتاج إلى تقليل وظائف التسجيل). مقاييسنا النهائية: RMSE=np.sqrt(-neg_MSE)



لذا فإن خط الأساس لدينا هو RMSE = 702 68 دولارًا أمريكيًا، وسنحاول تحسين هذه النتائج باستخدام كل ما اكتشفناه خلال مرحلة تحليل البيانات.
سنقوم بالخطوات التالية:
    - استخدام ميزات تحجيمها
    - إضافة ميزات السجل 
    - إضافة NaN ومقطع العمر للإشارة إلى الميزات
    - إضافة ميزات المسافة بين المدينة
    - إنشاء العديد من الميزات الجديدة
    - حاول التنبؤ بالسجل (الهدف) بدلاً من الهدف
    - ضبط بعض المعلمات الفائقة للنموذج



مرة أخرى، سيتم إجراء الجزء الأكبر من تعديل المعلمات الفائقة لاحقًا بعد إضافة بعض الميزات الجديدة. في الواقع، تتم عملية التحقق من الصحة وضبط المعلمات من خلال الأجزاء 8-11. 


In [ ]:
# using scaled data
X=pd.concat([train_df[dummies_names], X_train_scaled[numerical_features]], axis=1, ignore_index = True)
cv_scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1)
print(np.sqrt(-cv_scores.mean()))

In [ ]:
# adding NaN indicating feature
X=pd.concat([train_df[dummies_names+['total_bedrooms_is_nan']],
             X_train_scaled[numerical_features]], axis=1, ignore_index = True)
cv_scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1)
print(np.sqrt(-cv_scores.mean()))

In [ ]:
# adding house age cliiping indicating feature
X=pd.concat([train_df[dummies_names+['age_clipped']],
             X_train_scaled[numerical_features]], axis=1, ignore_index = True)
cv_scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1)
print(np.sqrt(-cv_scores.mean()))

In [ ]:
# adding log features
X=pd.concat([train_df[dummies_names+['age_clipped']], X_train_scaled[numerical_features+log_numerical_features]],
            axis=1, ignore_index = True)
cv_scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1)
print(np.sqrt(-cv_scores.mean()))

In [ ]:
# adding city distance features
X=pd.concat([train_df[dummies_names+['age_clipped']], X_train_scaled],
            axis=1, ignore_index = True)
cv_scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1)
print(np.sqrt(-cv_scores.mean()))


حتى هذه اللحظة حصلنا على أفضل النتائج باستخدام الميزات الرقمية + سجلاتها + age_clipped + المتغيرات الوهمية + المسافات إلى أكبر المدن.
دعونا نحاول إنشاء ميزات جديدة



## 9. إنشاء ميزات جديدة ووصف هذه العملية



في السابق، قمنا بالفعل بإنشاء وشرح الأساس المنطقي لإنشاء الميزات الجديدة. الآن دعونا ننشئ المزيد منها 



تعمل ميزات المسافات بين المدن، ولكن ربما توجد أيضًا بعض التبعيات غير الخطية بينها وبين المتغيرات المستهدفة.


In [ ]:
sns.set()
sns.pairplot(train_df[['distance_to_SF','distance_to_LA','median_house_value_log']])


بصريًا ليس واضحًا، لذا دعونا نحاول إنشاء متغيرين جديدين والتحقق من:


In [ ]:
new_features_train_df=pd.DataFrame(index=train_df.index)
new_features_test_df=pd.DataFrame(index=test_df.index)


new_features_train_df['1/distance_to_SF']=1/(train_df['distance_to_SF']+0.001)
new_features_train_df['1/distance_to_LA']=1/(train_df['distance_to_LA']+0.001)
new_features_train_df['log_distance_to_SF']=np.log1p(train_df['distance_to_SF'])
new_features_train_df['log_distance_to_LA']=np.log1p(train_df['distance_to_LA'])

new_features_test_df['1/distance_to_SF']=1/(test_df['distance_to_SF']+0.001)
new_features_test_df['1/distance_to_LA']=1/(test_df['distance_to_LA']+0.001)
new_features_test_df['log_distance_to_SF']=np.log1p(test_df['distance_to_SF'])
new_features_test_df['log_distance_to_LA']=np.log1p(test_df['distance_to_LA'])

يمكننا أيضًا إنشاء بعض الميزات المرتبطة بالازدهار:
- الغرف/الشخص - كم عدد الغرف الموجودة للشخص الواحد. كلما ارتفع عدد الأشخاص الأكثر ثراءً الذين يعيشون هناك - كلما زاد سعر المنازل التي يشترونها
- الغرف/المنزل - كم عدد الغرف الموجودة لكل أسرة. نفس الشيء ولكنه يتوافق مع عدد الغرف لكل أسرة (بافتراض الأسرة ~ الأسرة)، وليس لكل شخص.
- ميزتان متشابهتان ولكن مع احتساب غرف النوم فقط


In [ ]:
new_features_train_df['rooms/person']=train_df['total_rooms']/train_df['population']
new_features_train_df['rooms/household']=train_df['total_rooms']/train_df['households']

new_features_test_df['rooms/person']=test_df['total_rooms']/test_df['population']
new_features_test_df['rooms/household']=test_df['total_rooms']/test_df['households']


new_features_train_df['bedrooms/person']=train_df['total_bedrooms']/train_df['population']
new_features_train_df['bedrooms/household']=train_df['total_bedrooms']/train_df['households']

new_features_test_df['bedrooms/person']=test_df['total_bedrooms']/test_df['population']
new_features_test_df['bedrooms/household']=test_df['total_bedrooms']/test_df['households']


- يمكن تمييز فخامة المنزل بشراء عدد غرف النوم لكل غرفة


In [ ]:
new_features_train_df['bedroom/rooms']=train_df['total_bedrooms']/train_df['total_rooms']
new_features_test_df['bedroom/rooms']=test_df['total_bedrooms']/test_df['total_rooms']


- يمكن أن يكون متوسط عدد الأشخاص في الأسرة الواحدة إشارة إلى الرخاء أو في نفس الوقت إشارة إلى الثراء ولكن على أية حال يمكن أن يكون ميزة مفيدة


In [ ]:
new_features_train_df['average_size_of_household']=train_df['population']/train_df['households']
new_features_test_df['average_size_of_household']=test_df['population']/test_df['households']


وأخيرا دعونا نوسع نطاق كل هذه الميزات


In [ ]:
new_features_train_df=pd.DataFrame(scaler.fit_transform(new_features_train_df),
                            columns=new_features_train_df.columns, index=new_features_train_df.index)

new_features_test_df=pd.DataFrame(scaler.transform(new_features_test_df),
                            columns=new_features_test_df.columns, index=new_features_test_df.index)

In [ ]:
new_features_train_df.head()

In [ ]:
new_features_test_df.head()


سنضيف ميزات جديدة واحدة تلو الأخرى ونحتفظ فقط بالميزات التي تعمل على تحسين أفضل درجاتنا


In [ ]:
# computing current best score

X=pd.concat([train_df[dummies_names+['age_clipped']], X_train_scaled],
            axis=1, ignore_index = True)

cv_scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1)
best_score = np.sqrt(-cv_scores.mean())
print("Best score: ", best_score)

# list of the new good features
new_features_list=[]

for feature in new_features_train_df.columns:
    new_features_list.append(feature)
    X=pd.concat([train_df[dummies_names+['age_clipped']], X_train_scaled,
                 new_features_train_df[new_features_list]
                ],
                axis=1, ignore_index = True)
    cv_scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1)
    score = np.sqrt(-cv_scores.mean())
    if score >= best_score:
        new_features_list.remove(feature)
        print(feature, ' is not a good feature')
    else:
        print(feature, ' is a good feature')
        print('New best score: ', score)
        best_score=score


لقد حصلنا على 5 ميزات جيدة جديدة. دعونا نقوم بتحديث المتغير X الخاص بنا


In [ ]:
X=pd.concat([train_df[dummies_names+['age_clipped']], X_train_scaled,
             new_features_train_df[new_features_list]
            ],
            axis=1).reset_index(drop=True)
y=train_df['median_house_value'].reset_index(drop=True)


للتعامل مع سجل الهدف، نحتاج إلى إنشاء التحقق المتبادل الخاص بنا أو نموذج التنبؤ الخاص بنا. سنحاول الخيار الأول


In [ ]:
from sklearn.metrics import mean_squared_error

def cross_val_score_with_log(model=model, X=X,y=y,kf=kf, use_log=False):

    X_temp=np.array(X)

    # if use_log parameter is true we will predict log(y+1)
    if use_log:
        y_temp=np.log1p(y)
    else:
        y_temp=np.array(y)
    
    cv_scores=[]
    for train_index, test_index in kf.split(X_temp,y_temp):

        prediction = model.fit(X_temp[train_index], y_temp[train_index]).predict(X_temp[test_index])
        
        # if use_log parameter is true we should come back to the initial targer
        if use_log:
            prediction=np.expm1(prediction)
        cv_scores.append(-mean_squared_error(y[test_index],prediction))

    return np.sqrt(-np.mean(cv_scores))

In [ ]:
cross_val_score_with_log(X=X,y=y,kf=kf, use_log=False)


لقد حصلنا على نفس النتيجة تمامًا كما هو الحال مع وظيفة cross_val_score. وهذا يعني أن كل شيء يعمل بشكل جيد. الآن دعونا نحاول ضبط use_log على القيمة true


In [ ]:
cross_val_score_with_log(X=X,y=y,kf=kf, use_log=True)


لسوء الحظ، فإنه لم يساعد. لذلك سوف نلتزم بالإصدار السابق.
والآن سنقوم بضبط المعلمة الفائقة الوحيدة ذات المعنى لانحدار ريدج - ألفا.



## 10. رسم منحنيات التدريب والتحقق من الصحة



دعونا نرسم منحنى التحقق من الصحة


In [ ]:
from sklearn.model_selection import validation_curve

Cs=np.logspace(-5, 4, 10)
train_scores, valid_scores = validation_curve(model, X, y, "alpha", 
                                              Cs, cv=kf, scoring='neg_mean_squared_error')

plt.plot(Cs, np.sqrt(-train_scores.mean(axis=1)), 'ro-')

plt.fill_between(x=Cs, y1=np.sqrt(-train_scores.max(axis=1)), 
                 y2=np.sqrt(-train_scores.min(axis=1)), alpha=0.1, color = "red")


plt.plot(Cs, np.sqrt(-valid_scores.mean(axis=1)), 'bo-')

plt.fill_between(x=Cs, y1=np.sqrt(-valid_scores.max(axis=1)), 
                 y2=np.sqrt(-valid_scores.min(axis=1)), alpha=0.1, color = "blue")

plt.xscale('log')
plt.xlabel('alpha')
plt.ylabel('RMSE')
plt.title('Regularization Parameter Tuning')

plt.show()

In [ ]:
Cs[np.sqrt(-valid_scores.mean(axis=1)).argmin()]

يمكننا أن نرى أن منحنيات القطار والسيرة الذاتية قريبتان جدًا من بعضهما البعض، وهذه علامة على عدم الملاءمة. لا يتغير الفرق بين المنحنيات مع التغيير في ألفا، مما يعني أنه يجب علينا تجربة نماذج أكثر تعقيدًا مقارنة بالانحدار الخطي أو إضافة المزيد من الميزات الجديدة (على سبيل المثال، متعددة الحدود).
باستخدام هذا المنحنى يمكننا إيجاد القيمة المثلى للألفا. هو ألفا = 1. لكن في الواقع، لا يتغير توقعنا عندما تنخفض ألفا إلى أقل من 1.
دعونا نستخدم alpha=1 ونرسم منحنى التعلم


In [ ]:
from sklearn.model_selection import learning_curve

model=Ridge(alpha=1.0)

train_sizes, train_scores, valid_scores = learning_curve(model, X, y, train_sizes=list(range(50,10001,100)),
                                                         scoring='neg_mean_squared_error', cv=5)

plt.plot(train_sizes, np.sqrt(-train_scores.mean(axis=1)), 'ro-')

plt.fill_between(x=train_sizes, y1=np.sqrt(-train_scores.max(axis=1)), 
                 y2=np.sqrt(-train_scores.min(axis=1)), alpha=0.1, color = "red")

plt.plot(train_sizes, np.sqrt(-valid_scores.mean(axis=1)), 'bo-')

plt.fill_between(x=train_sizes, y1=np.sqrt(-valid_scores.max(axis=1)), 
                 y2=np.sqrt(-valid_scores.min(axis=1)), alpha=0.1, color = "blue")

plt.xlabel('Train size')
plt.ylabel('RMSE')
plt.title('Regularization Parameter Tuning')

plt.show()


تشير منحنيات التعلم إلى انحياز كبير للنموذج - وهذا يعني أننا لن نحسن نموذجنا بإضافة المزيد من البيانات، ولكن يمكننا محاولة استخدام نماذج أكثر تعقيدًا أو إضافة المزيد من الميزات لتحسين النتائج.
هذه النتيجة تتماشى مع نتائج منحنى التحقق من الصحة. لذلك دعونا ننتقل إلى النماذج الأكثر تعقيدا.



### غابة عشوائية



في الواقع، يمكننا فقط وضع جميع ميزاتنا في النموذج، ولكن يمكننا بسهولة تحسين الأداء الحسابي للنماذج المبنية على الشجرة، عن طريق حذف جميع المشتقات الرتيبة للميزات لأنها لا تساعد على الإطلاق.
على سبيل المثال، إضافة سجل (ميزة) لا يساعد النموذج القائم على الشجرة، بل سيجعله أكثر كثافة من الناحية الحسابية.
لذلك دعونا ندرب مصنف الغابة العشوائي بناءً على مجموعة مختصرة من الميزات


In [ ]:
X.columns

In [ ]:
features_for_trees=['INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN', 'age_clipped',
       'longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'distance_to_SF', 'distance_to_LA','bedroom/rooms']       

In [ ]:
%%time
from sklearn.ensemble import RandomForestRegressor

X_trees=X[features_for_trees]

model_rf=RandomForestRegressor(n_estimators=100, random_state=17)
cv_scores = cross_val_score(model_rf, X_trees, y, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1)

print(np.sqrt(-cv_scores.mean()))


يمكننا أن نرى تحسنا كبيرا، مقارنة بالنموذج الخطي ومن المحتمل أن يساعد المقدر الأعلى. لكن أولاً، دعونا نحاول ضبط المعلمات الفائقة الأخرى:


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid={'n_estimators': [100],
            'max_depth':  [22, 23, 24, 25],
            'max_features': [5,6,7,8]}

gs=GridSearchCV(model_rf, param_grid, scoring='neg_mean_squared_error', fit_params=None, n_jobs=-1, cv=kf, verbose=1)

gs.fit(X_trees,y)

In [ ]:
print(np.sqrt(-gs.best_score_))

In [ ]:
gs.best_params_

In [ ]:
best_depth=gs.best_params_['max_depth']
best_features=gs.best_params_['max_features']

In [ ]:
%%time
model_rf=RandomForestRegressor(n_estimators=100, max_depth=best_depth, max_features=best_features, random_state=17)
cv_scores = cross_val_score(model_rf, X_trees, y, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1)

print(np.sqrt(-cv_scores.mean()))


وبفضل الجهد الصغير نسبيًا، حصلنا على تحسن كبير في النتائج. يمكن تحسين نتائج الغابة العشوائية بشكل أكبر من خلال استخدام مقدرات n_estimators أعلى، فلنجد n_estimators عند استقرار النتائج. 


In [ ]:
model_rf=RandomForestRegressor(n_estimators=200,  max_depth=best_depth, max_features=best_features, random_state=17)
Cs=list(range(20,201,20))
train_scores, valid_scores = validation_curve(model_rf, X_trees, y, "n_estimators", 
                                              Cs, cv=kf, scoring='neg_mean_squared_error')

plt.plot(Cs, np.sqrt(-train_scores.mean(axis=1)), 'ro-')

plt.fill_between(x=Cs, y1=np.sqrt(-train_scores.max(axis=1)), 
                 y2=np.sqrt(-train_scores.min(axis=1)), alpha=0.1, color = "red")


plt.plot(Cs, np.sqrt(-valid_scores.mean(axis=1)), 'bo-')

plt.fill_between(x=Cs, y1=np.sqrt(-valid_scores.max(axis=1)), 
                 y2=np.sqrt(-valid_scores.min(axis=1)), alpha=0.1, color = "blue")

plt.xlabel('n_estimators')
plt.ylabel('RMSE')
plt.title('Regularization Parameter Tuning')

plt.show()

هذه المرة يمكننا أن نرى أن نتائج التدريب أفضل بكثير من السيرة الذاتية، لكنها لا بأس بها تمامًا بالنسبة للغابة العشوائية. 
القيمة الأعلى لـ n_estimators (> 100) لا تساعد كثيرًا. دعنا نلتزم بـ n_estimators=200 - فهو مرتفع بدرجة كافية ولكنه ليس مكثفًا حسابيًا للغاية.



### تعزيز التدرج



وأخيرا سنحاول استخدام LightGBM لحل مشكلتنا.
سنجرب النموذج خارج الصندوق، ثم نضبط بعض معلماته باستخدام البحث العشوائي


In [ ]:
# uncomment to install if you have not yet
#!pip install lightgbm

In [ ]:
%%time
from lightgbm.sklearn import LGBMRegressor

model_gb=LGBMRegressor()
cv_scores = cross_val_score(model_gb, X_trees, y, cv=kf, scoring='neg_mean_squared_error', n_jobs=1)

print(np.sqrt(-cv_scores.mean()))


يحتوي LGBMRegressor على معلمات تشعبية أكثر بكثير من النماذج السابقة. وبقدر ما تكون هذه مشكلة تعليمية، فلن نقضي الكثير من الوقت في ضبطها جميعًا. في هذه الحالة، يمكن لـ RandomizedSearchCV أن يمنحنا نتيجة جيدة جدًا بسرعة كبيرة، أسرع بكثير من GridSearch. سنجري التحسين في خطوتين: تحسين تعقيد النموذج وتحسين التقارب. دعونا نفعل ذلك.


In [ ]:
gs

In [ ]:
# model complexity optimization
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_grid={'max_depth':  randint(6,11),
            'num_leaves': randint(7,127),
            'reg_lambda': np.logspace(-3,0,100),
            'random_state': [17]}

gs=RandomizedSearchCV(model_gb, param_grid, n_iter = 50, scoring='neg_mean_squared_error', fit_params=None, 
                n_jobs=-1, cv=kf, verbose=1, random_state=17)

gs.fit(X_trees,y)

In [ ]:
np.sqrt(-gs.best_score_)

In [ ]:
gs.best_params_


دعونا نصلح n_estimators=500، فهو كبير بما يكفي ولكنه ليس مكثفًا حسابيًا بعد، ونبحث عن أفضل قيمة لمعدل_التعلم


In [ ]:
# model convergency optimization

param_grid={'n_estimators': [500],
            'learning_rate': np.logspace(-4, 0, 100),
            'max_depth':  [10],
            'num_leaves': [72],
            'reg_lambda': [0.0010722672220103231],
            'random_state': [17]}

gs=RandomizedSearchCV(model_gb, param_grid, n_iter = 20, scoring='neg_mean_squared_error', fit_params=None, 
                n_jobs=-1, cv=kf, verbose=1, random_state=17)

gs.fit(X_trees,y)

In [ ]:
np.sqrt(-gs.best_score_)

In [ ]:
gs.best_params_


لقد حصلنا على أفضل المعلمات لتعزيز التدرج وسوف نستخدمها للتنبؤ النهائي.



## 11. التنبؤ بالعينات الاختبارية أو المحتجزة



دعونا نلخص نتائج مشروعنا. سوف نقوم بحساب RMSE على مجموعة التحقق من الصحة والرفض ومقارنتها.


In [ ]:
results_df=pd.DataFrame(columns=['model','CV_results', 'holdout_results'])

In [ ]:
# hold-out features and target 
X_ho=pd.concat([test_df[dummies_names+['age_clipped']], X_test_scaled,
             new_features_test_df[new_features_list]],axis=1).reset_index(drop=True)
y_ho=test_df['median_house_value'].reset_index(drop=True)

X_trees_ho=X_ho[features_for_trees]

In [ ]:
%%time

#linear model
model=Ridge(alpha=1.0)

cv_scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1)
score_cv=np.sqrt(-np.mean(cv_scores.mean()))


prediction_ho = model.fit(X, y).predict(X_ho)
score_ho=np.sqrt(mean_squared_error(y_ho,prediction_ho))

results_df.loc[results_df.shape[0]]=['Linear Regression',  score_cv,  score_ho]

In [ ]:
%%time

#Random Forest
model_rf=RandomForestRegressor(n_estimators=200,  max_depth=23, max_features=5, random_state=17)

cv_scores = cross_val_score(model_rf, X_trees, y, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1)
score_cv=np.sqrt(-np.mean(cv_scores.mean()))


prediction_ho = model_rf.fit(X_trees, y).predict(X_trees_ho)
score_ho=np.sqrt(mean_squared_error(y_ho,prediction_ho))

results_df.loc[results_df.shape[0]]=['Random Forest',  score_cv,  score_ho]

In [ ]:
%%time

#Gradient boosting
model_gb=LGBMRegressor(reg_lambda=0.0010722672220103231, max_depth=10,
                       n_estimators=500, num_leaves=72, random_state=17, learning_rate=0.06734150657750829)
cv_scores = cross_val_score(model_gb, X_trees, y, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1)
score_cv=np.sqrt(-np.mean(cv_scores.mean()))

prediction_ho = model_gb.fit(X_trees, y).predict(X_trees_ho)
score_ho=np.sqrt(mean_squared_error(y_ho,prediction_ho))

results_df.loc[results_df.shape[0]]=['Gradient boosting',  score_cv,  score_ho]

In [ ]:
results_df


يبدو أننا قمنا بعمل جيد جدًا. تتوافق نتائج التحقق المتقاطع مع نتائج الرفض. أفضل نموذج للسيرة الذاتية لدينا - تعزيز التدرج، تبين أنه الأفضل في مجموعة البيانات المعلقة أيضًا (وهو أيضًا أسرع من الغابة العشوائية).



## 12. الاستنتاجات


لتلخيص ذلك، لدينا الحل الذي يمكنه التنبؤ بمتوسط ​​قيمة المنزل في الكتلة باستخدام RMSE \$46k using our best model - LGB. It is not an extremely precise prediction: \$46k وهو حوالي 20% من متوسط ​​سعر المنزل، ولكن يبدو أنه قريب من الحل المحتمل لهذه الفئات من النماذج بناءً على هذه البيانات (إنها مجموعة بيانات شائعة لكنني لم أجد أي حل بنتائج أفضل بكثير). 
لقد استخدمنا بيانات كاليفورنيا القديمة منذ عام 1990، لذا فهي ليست مفيدة الآن. ولكن من الممكن استخدام نفس النهج للتنبؤ بأسعار المساكن الحديثة (إذا تم تطبيقه على بيانات السوق الحالية).



لقد فعلنا الكثير ولكن من المؤكد أن النتائج يمكن تحسينها، على الأقل يمكن للمرء أن يحاول:
- هندسة الميزات: ميزات متعددة الحدود، مسافات أفضل للمدن (ليست إقليدية، تمثيل القطع الناقص للمدن)، متوسط قيم الهدف لأقرب الجيران جغرافيًا (يتطلب وظيفة تقدير مخصصة للتحقق من صحة التقاطع الصحيح)
- PCA لتقليل الأبعاد (لقد ذكرت ذلك ولكن لم أستخدمه)
- نماذج أخرى (على الأقل يمكن تجربة KNN وSVM بناءً على البيانات)
- يمكن إنفاق المزيد من الوقت والجهد على ضبط معلمات RF وLGB